# 🏥 E-Consultation Sentiment Analysis
### Deep Learning Mini Project — Bidirectional LSTM

**Pipeline:**
1. Data Ingestion (IMDB Dataset)
2. Exploratory Data Analysis
3. Text Preprocessing
4. Baseline Model (TF-IDF + Logistic Regression)
5. Deep Learning Model (BiLSTM)
6. Evaluation (Accuracy, F1, Confusion Matrix, ROC-AUC)
7. Model Comparison

---
**VTU-CMRIT | 2026**

## 0. Setup & Installations

In [ ]:
# Install required packages
!pip install -q datasets wordcloud plotly streamlit

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
from IPython.display import display

import sklearn
import tensorflow as tf

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU Available      : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Scikit-learn       : {sklearn.__version__}')

## 1. Data Ingestion

In [ ]:
from datasets import load_dataset

print('Loading IMDB dataset from HuggingFace...')
dataset = load_dataset('imdb')

train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])
df = pd.concat([train_df, test_df], ignore_index=True)
df.rename(columns={'text': 'comment', 'label': 'sentiment'}, inplace=True)
df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'positive'})

print(f'Total samples: {len(df):,}')
print(f'Class distribution:\n{df["sentiment"].value_counts()}')
df.head(3)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
PALETTE = {'positive': '#2ecc71', 'negative': '#e74c3c'}

# ── 2.1 Class Distribution ──────────────────
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['sentiment'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[PALETTE[s] for s in counts.index],
              edgecolor='white', width=0.4)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,} ({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Class Distribution — Sentiment Labels', fontsize=13)
ax.set_xlabel('Sentiment'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()
print('✅ Dataset is perfectly balanced!')

In [ ]:
# ── 2.2 Comment Length Distribution ─────────
df['word_count'] = df['comment'].str.split().str.len()
df['char_count'] = df['comment'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ['char_count', 'word_count'], ['Character Length', 'Word Count']):
    for sentiment, color in PALETTE.items():
        ax.hist(df[df['sentiment']==sentiment][col], bins=60,
                alpha=0.65, color=color, label=sentiment.capitalize())
    ax.set_xlabel(label); ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {label}')
    ax.legend()
    ax.set_xlim(0, df[col].quantile(0.98))
fig.suptitle('Comment Length Analysis', fontsize=14)
plt.tight_layout(); plt.show()

print(f'Average word count : {df["word_count"].mean():.1f}')
print(f'Median word count  : {df["word_count"].median():.0f}')
print(f'Max word count     : {df["word_count"].max():,}')

In [ ]:
# ── 2.3 Word Clouds ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sentiment, cmap in zip(axes, ['positive', 'negative'], ['Greens', 'Reds']):
    text = ' '.join(df[df['sentiment']==sentiment]['comment'].fillna(''))
    wc = WordCloud(width=700, height=400, background_color='#0d1117',
                   colormap=cmap, max_words=150).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sentiment.capitalize()} Comments', fontsize=13)
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Word Clouds by Sentiment Class', fontsize=15, color='white')
plt.tight_layout(); plt.show()

## 3. Text Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
MAX_VOCAB_SIZE = 20_000
MAX_SEQ_LEN   = 250
RANDOM_SEED   = 42

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)        # Remove HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text) # Remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)       # Keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Cleaning text...')
df['cleaned_comment'] = df['comment'].apply(clean_text)
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1})

# Remove empty rows post-cleaning
df = df[df['cleaned_comment'].str.len() > 2].reset_index(drop=True)
print(f'Samples after cleaning: {len(df):,}')
df[['comment', 'cleaned_comment', 'label']].head(3)

In [ ]:
# ── Stratified Train/Val/Test Split (80/10/10) ──
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=RANDOM_SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=RANDOM_SEED)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Train class balance: {train_df["label"].value_counts().to_dict()}')

# ── Tokenization ────────────────────────────────
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['cleaned_comment'])  # Fit ONLY on train
print(f'Vocabulary size: {min(len(tokenizer.word_index)+1, MAX_VOCAB_SIZE):,}')

# ── Padding ──────────────────────────────────────
def to_padded(texts):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

X_train = to_padded(train_df['cleaned_comment'])
X_val   = to_padded(val_df['cleaned_comment'])
X_test  = to_padded(test_df['cleaned_comment'])

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape  : {X_val.shape}')
print(f'X_test shape : {X_test.shape}')

## 4. Baseline Model — TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Build TF-IDF features
vectorizer = TfidfVectorizer(max_features=20_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
X_train_tfidf = vectorizer.fit_transform(train_df['cleaned_comment'])
X_val_tfidf   = vectorizer.transform(val_df['cleaned_comment'])
X_test_tfidf  = vectorizer.transform(test_df['cleaned_comment'])

print(f'TF-IDF feature size: {X_train_tfidf.shape[1]:,}')

# Train Logistic Regression
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED)
lr_model.fit(X_train_tfidf, y_train)

# Evaluate
lr_pred = lr_model.predict(X_test_tfidf)
lr_acc  = accuracy_score(y_test, lr_pred)
lr_f1   = f1_score(y_test, lr_pred, average='weighted')

print(f'\n[BASELINE] Test Accuracy : {lr_acc*100:.2f}%')
print(f'[BASELINE] Test F1 Score : {lr_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Baseline Confusion Matrix
cm = confusion_matrix(y_test, lr_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'],
            ax=ax, annot_kws={'size': 14, 'weight': 'bold'})
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Baseline (LR)', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Deep Learning Model — Bidirectional LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout,
    SpatialDropout1D, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Build Model
model = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE + 1, output_dim=128, input_length=MAX_SEQ_LEN, name='Embeddings'),
    SpatialDropout1D(0.25),
    Bidirectional(LSTM(128, return_sequences=True, recurrent_dropout=0.1), name='BiLSTM'),
    LSTM(64, recurrent_dropout=0.1, name='LSTM'),
    BatchNormalization(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid'),
], name='BiLSTM_Sentiment_Classifier')

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_bilstm_model.h5', monitor='val_loss', save_best_only=True, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs_range, history.history['loss'],     'b-o', label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'], 'r-o', label='Val Loss')
axes[0].set_title('Loss Curves'); axes[0].legend()
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(epochs_range, history.history['accuracy'],     'b-s', label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-s', label='Val Acc')
axes[1].set_title('Accuracy Curves'); axes[1].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

fig.suptitle('BiLSTM — Training vs. Validation Curves', fontsize=14)
plt.tight_layout(); plt.show()

## 6. Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve
)

# Predictions
y_prob = model.predict(X_test, verbose=1).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
dl_acc       = accuracy_score(y_test, y_pred)
dl_precision = precision_score(y_test, y_pred, average='weighted')
dl_recall    = recall_score(y_test, y_pred, average='weighted')
dl_f1        = f1_score(y_test, y_pred, average='weighted')
dl_auc       = roc_auc_score(y_test, y_prob)

print('=' * 55)
print('  BILSTM MODEL — TEST SET EVALUATION')
print('=' * 55)
print(f'  Accuracy  : {dl_acc*100:.2f}%')
print(f'  Precision : {dl_precision:.4f}')
print(f'  Recall    : {dl_recall:.4f}')
print(f'  F1 Score  : {dl_f1:.4f}')
print(f'  ROC-AUC   : {dl_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'],
            ax=axes[0], annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix — BiLSTM', fontsize=13)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'BiLSTM (AUC = {dl_auc:.4f})')
axes[1].plot([0,1], [0,1], 'k--', linewidth=1.2, label='Random (AUC = 0.50)')
axes[1].fill_between(fpr, tpr, alpha=0.12, color='blue')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC-AUC Curve', fontsize=13)
axes[1].legend()

plt.tight_layout(); plt.show()

## 7. Model Comparison — Baseline vs. BiLSTM

In [ ]:
from sklearn.metrics import precision_score as ps, recall_score as rs

metrics_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
baseline_scores = [
    accuracy_score(y_test, lr_pred),
    ps(y_test, lr_pred, average='weighted'),
    rs(y_test, lr_pred, average='weighted'),
    f1_score(y_test, lr_pred, average='weighted'),
]
dl_scores = [dl_acc, dl_precision, dl_recall, dl_f1]

x     = np.arange(len(metrics_labels))
width = 0.30

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline (TF-IDF + LR)', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, dl_scores,       width, label='BiLSTM (Deep Learning)', color='#e67e22', alpha=0.85)

for bars, scores in [(bars1, baseline_scores), (bars2, dl_scores)]:
    for bar, s in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{s:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_ylabel('Score'); ax.set_ylim(0.5, 1.05)
ax.set_title('Model Comparison — Baseline vs. Deep Learning', fontsize=13)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.axhline(y=0.9, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout(); plt.show()

print(f'\nImprovement over baseline:')
for label, base, dl in zip(metrics_labels, baseline_scores, dl_scores):
    print(f'  {label:<12}: +{(dl - base)*100:.2f}%')

## 8. Single Comment Inference (Demo)
Test the model on custom E-consultation comments.

In [ ]:
def predict_comment(comment_text):
    cleaned = clean_text(comment_text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
    prob    = float(model.predict(padded, verbose=0)[0][0])
    label   = 'POSITIVE 😊' if prob >= 0.5 else 'NEGATIVE 😞'
    conf    = prob if prob >= 0.5 else 1 - prob
    print(f'Comment   : {comment_text[:80]}...')
    print(f'Sentiment : {label}')
    print(f'Confidence: {conf*100:.1f}%')
    print(f'Pos. Score: {prob*100:.2f}%')
    print('-' * 60)

test_comments = [
    'The doctor was very attentive and explained everything clearly. Excellent service!',
    'I waited 2 hours and still could not see anyone. Very disappointing experience.',
    'The online consultation was convenient and the response was quick.',
    'The portal was confusing and I could not upload my documents properly.',
]

for comment in test_comments:
    predict_comment(comment)

In [ ]:
# Save model for Streamlit app
model.save('bilstm_sentiment_model.h5')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print('Model and tokenizer saved!')
print('\nFinal Summary:')
print(f'  Baseline Accuracy : {accuracy_score(y_test, lr_pred)*100:.2f}%')
print(f'  BiLSTM Accuracy   : {dl_acc*100:.2f}%')
print(f'  BiLSTM F1         : {dl_f1:.4f}')
print(f'  BiLSTM ROC-AUC    : {dl_auc:.4f}')